# xBD Dataset Exploration

Exploratory data analysis of the xBD/xView2 disaster damage dataset.

This notebook covers:
- Loading and parsing xBD labels
- Visualizing building polygons and damage classes
- Rasterizing polygons to masks
- Tiling images for model training

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import cv2
from shapely.geometry import Polygon

from xbd_damage_assessment.data.label_parser import xBDLabelParser
from xbd_damage_assessment.data.rasterize import rasterize_building_masks, rasterize_damage_masks
from xbd_damage_assessment.data.tiling import create_tiles_with_overlap

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Sample Data

Load a sample disaster image and its labels.

In [ ]:
# TODO: Update these paths to point to your xBD data
data_root = Path('../data/raw/xbd/train')

# Find first disaster
sample_image = sorted((data_root / 'images').glob('*_pre_disaster.png'))[0]
disaster_id = sample_image.stem.replace('_pre_disaster', '')

print(f"Loading disaster: {disaster_id}")

# Load images
pre_image = cv2.imread(str(sample_image), cv2.IMREAD_COLOR)
pre_image = cv2.cvtColor(pre_image, cv2.COLOR_BGR2RGB)

post_image_path = data_root / 'images' / f'{disaster_id}_post_disaster.png'
post_image = cv2.imread(str(post_image_path), cv2.IMREAD_COLOR)
post_image = cv2.cvtColor(post_image, cv2.COLOR_BGR2RGB)

print(f"Image shape: {pre_image.shape}")

## 2. Parse Labels

Parse the JSON label file to extract building polygons and damage classes.

In [ ]:
label_path = data_root / 'labels' / f'{disaster_id}_pre_disaster.json'

parser = xBDLabelParser()
polygons, damage_classes, building_uids = parser.parse(label_path)

print(f"Found {len(polygons)} buildings")
print(f"Damage distribution: {dict(zip(*np.unique(damage_classes, return_counts=True)))}")

## 3. Visualize Polygons

In [ ]:
# TODO: Add visualization code here
# Plot pre/post images with building polygons colored by damage class

print("Visualization coming soon!")

## 4. Rasterize Masks

In [ ]:
# Create binary building mask
building_mask = rasterize_building_masks(polygons, pre_image.shape[:2])

# Create 4-class damage mask
damage_mask = rasterize_damage_masks(polygons, damage_classes, pre_image.shape[:2])

print(f"Building mask shape: {building_mask.shape}")
print(f"Building coverage: {np.sum(building_mask > 0) / building_mask.size * 100:.2f}%")
print(f"Damage mask classes: {np.unique(damage_mask)}")

## 5. Tiling

Split the large image into smaller patches for model training.

In [ ]:
# Create tiles
image_tiles, mask_tiles, tile_infos = create_tiles_with_overlap(
    pre_image,
    building_mask,
    tile_size=512,
    overlap=64,
    min_building_pixels=100
)

print(f"Created {len(image_tiles)} tiles")
print(f"Tile shape: {image_tiles[0].shape}")

## Next Steps

- Run full preprocessing pipeline: `python -m xbd_damage_assessment.data.preprocess`
- Explore augmentations
- Analyze class imbalance
- Visualize training samples